#LSTM (train + log to W&B ui)

This script follows the same Federated Learning (FL) logic as our previous MLP example but replaces the architecture with an LSTM (Long Short-Term Memory) network.

While LSTMs are typically used for sequential data (like text or time-series), this implementation adapts them for tabular data to see if the LSTM's internal "memory" cells can capture complex feature correlations better than a standard dense network.

In [ ]:
# =========================
# Approach 3 (LSTM) — FULL W&B UPDATED (round-wise eval + curves + tuned threshold)
# =========================

!pip -q install wandb scikit-learn imbalanced-learn kagglehub

import os, time
import numpy as np
import pandas as pd
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e20285 (e20285-university-of-peradeniya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
PROJECT = "fl-fraud-compare"
GROUP = os.environ.get("WANDB_GROUP", "compare4-" + str(int(time.time())))
print("W&B GROUP =", GROUP)

W&B GROUP = compare4-1766896616


In [ ]:
# -------------------------
# Config for this approach
# -------------------------
USE_SMOTE = False  # toggle if you want; will be logged to W&B

NUM_CLIENTS = 10
NUM_ROUNDS = 15
LOCAL_EPOCHS = 2
LR = 1e-3
BATCH_SIZE = 256
WEIGHT_DECAY = 1e-5

HIDDEN_DIM = 30
NUM_LAYERS = 1

DEFAULT_THRESHOLD = 0.5
THRESH_MIN, THRESH_MAX, THRESH_STEPS = 0.01, 0.99, 99
RANDOM_STATE = 42


In [ ]:
run = wandb.init(
    project=PROJECT,
    group=GROUP,
    job_type="train",
    name="Approach3-LSTM",
    config={
        "approach": "Approach3_LSTM",
        "model": "LSTMTabular(seq_len=1)",
        "dp": False,
        "use_smote": USE_SMOTE,

        "num_clients": NUM_CLIENTS,
        "num_rounds": NUM_ROUNDS,
        "local_epochs": LOCAL_EPOCHS,
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "weighted_fedavg": True,

        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,

        "default_threshold": DEFAULT_THRESHOLD,
        "tune_threshold": True,
        "threshold_min": THRESH_MIN,
        "threshold_max": THRESH_MAX,
        "threshold_steps": THRESH_STEPS,
        "random_state": RANDOM_STATE,

        "criterion": "BCEWithLogitsLoss(pos_weight if not SMOTE)"
    }
)


In [ ]:
# =========================
# 1) Load + split + scale + (optional) SMOTE (TRAIN ONLY)
# =========================
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Dataset path:", path)


Using Colab cache for faster access to the 'creditcardfraud' dataset.
Dataset path: /kaggle/input/creditcardfraud


In [ ]:
import glob
csvs = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
assert csvs, f"No CSV found under: {path}"
file_path = csvs[0]
print("Using CSV:", file_path)

Using CSV: /kaggle/input/creditcardfraud/creditcard.csv


In [ ]:
df = pd.read_csv(file_path)
df = df.drop_duplicates()
df = df.fillna(df.mean(numeric_only=True))

In [ ]:
X = df.drop("Class", axis=1).values
y = df["Class"].values.astype(int)

In [ ]:
# Split BEFORE scaling (no leakage)
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_raw
)

In [ ]:
# Scale using train stats only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)
X_test_scaled  = scaler.transform(X_test_raw)

In [ ]:
# Optional SMOTE (TRAIN only)
if USE_SMOTE:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(sampling_strategy="minority", random_state=RANDOM_STATE)
    X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train_raw)
else:
    X_train_final, y_train_final = X_train_scaled, y_train_raw

In [ ]:
print("Train counts:", np.bincount(y_train_final))
print("Val   counts:", np.bincount(y_val))
print("Test  counts:", np.bincount(y_test))

Train counts: [181282    302]
Val   counts: [45320    76]
Test  counts: [56651    95]


In [ ]:
# Log dataset info
run.summary["data/n_rows_total"] = int(len(df))
run.summary["data/n_features"] = int(X_train_final.shape[1])
run.summary["data/train_size"] = int(len(X_train_final))
run.summary["data/val_size"] = int(len(X_val_scaled))
run.summary["data/test_size"] = int(len(X_test_scaled))
run.summary["data/train_pos_rate"] = float(np.mean(y_train_final))
run.summary["data/val_pos_rate"] = float(np.mean(y_val))
run.summary["data/test_pos_rate"] = float(np.mean(y_test))

In [ ]:
# =========================
# 2) Create federated clients (from TRAIN only)
# =========================
client_data   = np.array_split(X_train_final, NUM_CLIENTS)
client_labels = np.array_split(y_train_final, NUM_CLIENTS)
client_sizes  = [len(cd) for cd in client_data]

In [ ]:
# =========================
# 3) LSTM model + FL helpers
# =========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [ ]:
input_dim = X_train_final.shape[1]

In [ ]:
class LSTMTabular(nn.Module):
    def __init__(self, input_dim=30, hidden_dim=30, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)  # logits

    def forward(self, x):
        # x: (batch, 1, input_dim)
        out, (h, c) = self.lstm(x)
        last_h = h[-1]            # (batch, hidden_dim)
        logits = self.fc(last_h)  # (batch, 1)
        return logits

In [ ]:
def make_bce_logits_loss(y_train_binary: np.ndarray):
    if USE_SMOTE:
        return nn.BCEWithLogitsLoss()

    pos = int((y_train_binary == 1).sum())
    neg = int((y_train_binary == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=device)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

criterion = make_bce_logits_loss(y_train_final)

In [ ]:
def local_train(client_x, client_y, model, epochs=2, lr=1e-3):
    model.train()
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    x = torch.tensor(client_x, dtype=torch.float32).unsqueeze(1)  # (N,1,input_dim)
    y = torch.tensor(client_y, dtype=torch.float32).view(-1, 1)

    ds = TensorDataset(x, y)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

    for _ in range(epochs):
        for bx, by in dl:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()

    return model

In [ ]:
def fedavg_weighted(models, weights):
    global_model = LSTMTabular(input_dim=input_dim, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS).to(device)
    total = float(sum(weights))

    with torch.no_grad():
        for gp in global_model.parameters():
            gp.zero_()

        for m, w in zip(models, weights):
            frac = float(w) / total
            for gp, cp in zip(global_model.parameters(), m.parameters()):
                gp.add_(cp.data.to(device) * frac)

    return global_model

In [ ]:
@torch.no_grad()
def predict_proba(model, X_np):
    model.eval()
    model.to(device)
    x = torch.tensor(X_np, dtype=torch.float32).unsqueeze(1).to(device)  # (N,1,input_dim)
    logits = model(x)
    probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
    return probs

In [ ]:
@torch.no_grad()
def evaluate_loss(model, X_np, y_np):
    model.eval()
    model.to(device)
    x = torch.tensor(X_np, dtype=torch.float32).unsqueeze(1).to(device)
    y = torch.tensor(y_np, dtype=torch.float32).view(-1, 1).to(device)
    logits = model(x)
    return float(criterion(logits, y).item())

In [ ]:
# =========================
# 4) Metrics for UI logging
# =========================
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, confusion_matrix
)

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    acc = float((y_pred == y_true).mean())

    roc_auc = None
    pr_auc = None
    if len(np.unique(y_true)) > 1:
        roc_auc = float(roc_auc_score(y_true, y_prob))
        pr_auc = float(average_precision_score(y_true, y_prob))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "acc": acc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "y_pred": y_pred
    }

In [ ]:
# =========================
# 5) Federated training + per-round W&B eval logging
# =========================
models = [LSTMTabular(input_dim=input_dim, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS) for _ in range(NUM_CLIENTS)]
for r in range(NUM_ROUNDS):
    print(f"\n=== Round {r+1}/{NUM_ROUNDS} ===")

    # local training
    for i in range(NUM_CLIENTS):
        models[i] = local_train(client_data[i], client_labels[i], models[i], epochs=LOCAL_EPOCHS, lr=LR)

    # aggregate
    global_model = fedavg_weighted(models, client_sizes)

    # broadcast
    gstate = global_model.state_dict()
    for i in range(NUM_CLIENTS):
        models[i].load_state_dict(gstate)

    # ---- Evaluate on VAL + TEST using DEFAULT threshold ----
    val_probs = predict_proba(global_model, X_val_scaled)
    test_probs = predict_proba(global_model, X_test_scaled)

    val_loss = evaluate_loss(global_model, X_val_scaled, y_val)
    test_loss = evaluate_loss(global_model, X_test_scaled, y_test)

    val_m = compute_metrics(y_val, val_probs, threshold=DEFAULT_THRESHOLD)
    test_m = compute_metrics(y_test, test_probs, threshold=DEFAULT_THRESHOLD)

    log_dict = {
        "round": r + 1,

        # validation
        "val/loss": val_loss,
        "val/acc": val_m["acc"],
        "val/precision": val_m["precision"],
        "val/recall": val_m["recall"],
        "val/f1": val_m["f1"],

        # test (default threshold)
        "test_default/loss": test_loss,
        "test_default/acc": test_m["acc"],
        "test_default/precision": test_m["precision"],
        "test_default/recall": test_m["recall"],
        "test_default/f1": test_m["f1"],
    }

    if val_m["roc_auc"] is not None:  log_dict["val/auc"] = val_m["roc_auc"]
    if val_m["pr_auc"] is not None:   log_dict["val/pr_auc"] = val_m["pr_auc"]
    if test_m["roc_auc"] is not None: log_dict["test_default/auc"] = test_m["roc_auc"]
    if test_m["pr_auc"] is not None:  log_dict["test_default/pr_auc"] = test_m["pr_auc"]

    wandb.log(log_dict, step=r)

print("\nFederated training complete.")


=== Round 1/15 ===

=== Round 2/15 ===

=== Round 3/15 ===

=== Round 4/15 ===

=== Round 5/15 ===

=== Round 6/15 ===

=== Round 7/15 ===

=== Round 8/15 ===

=== Round 9/15 ===

=== Round 10/15 ===

=== Round 11/15 ===

=== Round 12/15 ===

=== Round 13/15 ===

=== Round 14/15 ===

=== Round 15/15 ===

Federated training complete.


In [ ]:
# =========================
# 6) Threshold tuning on VAL + final TEST metrics (tuned)
# =========================
val_probs = predict_proba(global_model, X_val_scaled)
test_probs = predict_proba(global_model, X_test_scaled)

best_t, best_f1 = DEFAULT_THRESHOLD, -1.0
thresholds = np.linspace(THRESH_MIN, THRESH_MAX, THRESH_STEPS)

threshold_table = wandb.Table(columns=["threshold", "val_f1", "val_precision", "val_recall"])


In [ ]:
for t in thresholds:
    m = compute_metrics(y_val, val_probs, threshold=float(t))
    threshold_table.add_data(float(t), float(m["f1"]), float(m["precision"]), float(m["recall"]))
    if m["f1"] > best_f1:
        best_f1 = m["f1"]
        best_t = float(t)

wandb.log({"threshold_sweep": threshold_table})
run.summary["best_threshold/value"] = float(best_t)
run.summary["best_threshold/val_f1"] = float(best_f1)

In [ ]:
# Final TEST metrics with tuned threshold
final_test_loss = evaluate_loss(global_model, X_test_scaled, y_test)
final_test_m = compute_metrics(y_test, test_probs, threshold=best_t)

run.summary["final/test_loss"] = float(final_test_loss)
run.summary["final/test_acc"] = float(final_test_m["acc"])
run.summary["final/test_precision"] = float(final_test_m["precision"])
run.summary["final/test_recall"] = float(final_test_m["recall"])
run.summary["final/test_f1"] = float(final_test_m["f1"])
if final_test_m["roc_auc"] is not None:
    run.summary["final/test_auc"] = float(final_test_m["roc_auc"])
if final_test_m["pr_auc"] is not None:
    run.summary["final/test_pr_auc"] = float(final_test_m["pr_auc"])

In [ ]:
print("\n=== FINAL TEST METRICS (tuned threshold) ===")
print(f"Best threshold (VAL): {best_t:.2f} | VAL F1: {best_f1:.4f}")
print(f"Test Loss : {final_test_loss:.4f}")
print(f"Accuracy  : {final_test_m['acc']:.4f}")
print(f"F1        : {final_test_m['f1']:.4f}")
print(f"Precision : {final_test_m['precision']:.4f}")
print(f"Recall    : {final_test_m['recall']:.4f}")
print(f"AUC       : {final_test_m['roc_auc']}")
print(f"PR-AUC    : {final_test_m['pr_auc']}")


=== FINAL TEST METRICS (tuned threshold) ===
Best threshold (VAL): 0.99 | VAL F1: 0.8387
Test Loss : 0.4901
Accuracy  : 0.9994
F1        : 0.8111
Precision : 0.8588
Recall    : 0.7684
AUC       : 0.9634767444993306
PR-AUC    : 0.7015260628261153


In [ ]:
# =========================
# 7) W&B visual panels: CM + ROC + PR (TEST, tuned threshold)
# =========================
wandb.log({
    "confusion_matrix_test_tuned": wandb.plot.confusion_matrix(
        y_true=np.asarray(y_test).astype(int),
        preds=final_test_m["y_pred"],
        class_names=["legit", "fraud"]
    )
})

probas_2col = np.stack([1.0 - test_probs, test_probs], axis=1)

wandb.log({
    "roc_test": wandb.plot.roc_curve(
        np.asarray(y_test).astype(int),
        probas_2col,
        labels=["legit", "fraud"]
    )
})
try:
    wandb.log({
        "pr_test": wandb.plot.pr_curve(
            np.asarray(y_test).astype(int),
            probas_2col,
            labels=["legit", "fraud"]
        )
    })
except Exception as e:
    print("PR curve plot not available in this wandb version:", e)

run.finish()

round,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_default/acc,█▁▁▄▄▄▄▅▅▆▅▅▅▅▅
test_default/auc,▁▇██▇▇▇▇▆▇▆▆▆▆▆
test_default/f1,█▁▁▂▃▃▃▃▄▄▄▄▄▄▄
test_default/loss,█▄▂▁▁▁▁▁▁▁▁▁▁▁▁
test_default/pr_auc,▁▇▆▆▆▆▇▇▇████▆█
test_default/precision,█▁▁▂▂▃▃▃▃▄▃▄▄▄▄
test_default/recall,▁▆▇▅▅▃▇▆▅▅█████
val/acc,█▁▁▄▄▅▄▅▅▆▅▆▆▆▆
val/auc,▁▇▇████████████
+5,...
